# Installations


In [1]:
!pip install dash jupyter-dash
# Install ngrok
!pip install pyngrok

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 7.2/7.2 MB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 37.1 MB/s eta 0:00:00


# Imports


In [2]:
from dash import Dash, dcc, html, Input, Output
import plotly.express as px
import pandas as pd
from pyngrok import ngrok



# Get Necessary Files from gDrive

In [3]:
# Mount Drive instead of csv upload

from google.colab import drive
drive.mount('/content/drive')



Mounted at /content/drive


In [4]:

base_path = "/content/drive/MyDrive/fire_data/"

fires_over_time_pd = pd.read_csv(base_path + "fires_over_time.csv")
fires_by_period_pd = pd.read_csv(base_path + "fires_by_period.csv")
frp_stats_pd = pd.read_csv(base_path + "frp_stats.csv")
spatial_counts_pd = pd.read_csv(base_path + "spatial_counts.csv")
confidence_counts_pd = pd.read_csv(base_path + "confidence_counts.csv")
frp_by_confidence_pd = pd.read_csv(base_path + "frp_by_confidence.csv")

# dataset for dash
df = pd.read_csv(base_path + "small_dashboard_dataset.csv")



In [5]:
df["acq_date"] = pd.to_datetime(df["acq_date"])
df["lat_bin"] = df["latitude"].round(1)
df["lon_bin"] = df["longitude"].round(1)




In [6]:
df["month"] = df["acq_date"].dt.month

df["period"] = "Dec–Jan"

df.loc[df["month"].isin([2,3]), "period"] = "Feb–Mar"
df.loc[df["month"].isin([4,5]), "period"] = "Apr–May"
df.loc[df["month"].isin([6,7]), "period"] = "Jun–Jul"
df.loc[df["month"].isin([8,9]), "period"] = "Aug–Sep"
df.loc[df["month"].isin([10,11]), "period"] = "Oct–Nov"


In [7]:
# sample dataset for dashboard performance

df_dash = df

# Create Dash App

In [8]:
app = Dash(__name__)

In [9]:
# LAYOUT EDITS
from dash import Dash, dcc, html, Input, Output
import plotly.express as px

app = Dash(__name__)

# ---------- Initial Figures ----------

fig_map = px.density_mapbox(
    spatial_counts_pd,
    lat="lat_bin",
    lon="lon_bin",
    z="fire_count",
    radius=20,
    zoom=2,
    mapbox_style="open-street-map"
    #title="Global Wildfire Hotspots"
)

fig_time = px.line(
    fires_over_time_pd,
    x="acq_date",
    y="fire_count"
)

fig_season = px.bar(
    fires_by_period_pd,
    x="period",
    y="fire_count",
    title="Wildfire Activity by Seasonal Period",
    labels={
        "period": "Seasonal Period",
        "fire_count": "Number of Fire Detections"
    }
)

fig_frp = px.bar(
    frp_stats_pd,
    x="period",
    y="mean_frp",
    title="Average Fire Intensity by Season"
)

app.layout = html.Div([

    html.H1(
        "Global Wildfire Analysis Dashboard",
        style={
            "textAlign": "center",
            "marginBottom": "40px"
        }
    ),

    # -------- TOP ROW: CONTROLS + LINE CHART --------

    html.Div([

                # -------- LEFT PANEL --------
        html.Div([

            # -------- SUMMARY / KPIs --------
            html.H3("Summary"),

            html.Div(
                id="kpi-container",
                style={
                    "display": "flex",
                    "gap": "20px",
                    "marginTop": "15px",
                    "marginBottom": "25px"
                }
            ),

            html.Hr(),

            # -------- FILTERS --------
            html.H3("Filters"),

            html.Label("Confidence Level"),
            dcc.Dropdown(
                id="confidence-dropdown",
                options=[
                    {"label": "All", "value": "all"},
                    {"label": "Nominal", "value": "n"},
                    {"label": "High", "value": "h"}
                ],
                value="all",
                clearable=False
            ),

            html.Br(),

            html.Label("FRP Range"),
            dcc.RangeSlider(
                id="frp-slider",
                min=0,
                max=50,
                step=10,
                value=[0, 50],
                updatemode="mouseup"
            ),

            html.Br(),
            html.Br(),

            html.Label("Season"),
            dcc.Dropdown(
                id="period-dropdown",
                options=[
                    {"label": "All", "value": "all"},
                    {"label": "Dec–Jan", "value": "Dec–Jan"},
                    {"label": "Feb–Mar", "value": "Feb–Mar"},
                    {"label": "Apr–May", "value": "Apr–May"},
                    {"label": "Jun–Jul", "value": "Jun–Jul"},
                    {"label": "Aug–Sep", "value": "Aug–Sep"},
                    {"label": "Oct–Nov", "value": "Oct–Nov"}
                ],
                value="all",
                clearable=False
            )

        ], style={
            "width": "35%",
            "padding": "20px"
        })
        ,

        # -------- RIGHT PANEL: LINE CHART --------
        html.Div([

            html.H2(
                "Wildfire Activity Over Time",
                style={"textAlign": "center"}
            ),

            dcc.Graph(
                id="time-chart",
                figure=fig_time
            )

        ], style={
            "width": "65%"
        })

    ], style={
        "display": "flex",
        "gap": "30px",
        "width": "95%",
        "margin": "auto",
        "marginBottom": "40px",
        "alignItems": "flex-start"
    }),

    # -------- MAP SECTION --------

    html.H2(
        "Spatial Distribution of Wildfires",
        style={
            "textAlign": "center",
            "marginBottom": "10px"
        }
    ),

    html.Div(
        dcc.Graph(id="map-chart", figure=fig_map),
        style={
            "width": "95%",
            "margin": "auto",
            "marginBottom": "50px"
        }
    ),

    # -------- SEASONAL BENCHMARK SECTION --------

    html.H2(
        "Seasonal Wildfire Patterns (Full Dataset Benchmark)",
        style={
            "textAlign": "center",
            "marginBottom": "10px"
        }
    ),

    html.P(
        "These charts summarize the full dataset and remain constant to provide a benchmark for filtered exploration above.",
        style={
            "textAlign": "center",
            "marginBottom": "30px",
            "color": "gray"
        }
    ),

    html.Div([

        html.Div(
            dcc.Graph(id="season-chart", figure=fig_season),
            style={"width": "50%"}
        ),

        html.Div(
            dcc.Graph(id="frp-chart", figure=fig_frp),
            style={"width": "50%"}
        )

    ], style={
        "display": "flex",
        "width": "95%",
        "margin": "auto",
        "gap": "20px"
    })

])


# ---------- Callback ----------

@app.callback(
    Output("map-chart","figure"),
    Output("time-chart","figure"),
    Output("season-chart","figure"),
    Output("frp-chart","figure"),
    Output("kpi-container","children"),
    Input("confidence-dropdown","value"),
    Input("frp-slider","value"),
    Input("period-dropdown","value")
)

def update_dashboard(confidence, frp_range, period):

    df_filtered = df_dash

    # confidence filter
    if confidence != "all":
        df_filtered = df_filtered[df_filtered["confidence"] == confidence]

    # frp filter
    df_filtered = df_filtered[
        (df_filtered["frp"] >= frp_range[0]) &
        (df_filtered["frp"] <= frp_range[1])
    ]

    # season filter
    if period != "all":
        df_filtered = df_filtered[df_filtered["period"] == period]

    # ---------- MAP ----------

    spatial_counts = (
        df_filtered.groupby(["lat_bin","lon_bin"])
        .size()
        .reset_index(name="fire_count")
    )

    spatial_counts = spatial_counts.sort_values(
        "fire_count",
        ascending=False
    ).head(500)

    fig_map = px.density_mapbox(
        spatial_counts,
        lat="lat_bin",
        lon="lon_bin",
        z="fire_count",
        radius=20,
        zoom=2,
        mapbox_style="open-street-map",
        color_continuous_scale="YlOrRd",
        title="Global Wildfire Detection Density",
        labels= {"fire_count" : "Fire Count"}
    )

    fig_map.update_layout(
        margin=dict(l=0,r=0,t=30,b=0)
    )



    # ---------- TIME ----------

    fires_over_time = (
        df_filtered.groupby("acq_date")
        .size()
        .reset_index(name="fire_count")
    )

    fig_time = px.line(
        fires_over_time,
        x="acq_date",
        y="fire_count",
        title="Wildfire Detections Over Time",
        labels={
          "acq_date": "Date",
          "fire_count": "Number of Fire Detections"
    }
    )

    # ---------- SEASON ----------

    fires_by_period = (
        df.groupby("period")
        .size()
        .reset_index(name="fire_count")
    )

    period_order = [
        "Dec–Jan","Feb–Mar","Apr–May",
        "Jun–Jul","Aug–Sep","Oct–Nov"
    ]

    fires_by_period["period"] = pd.Categorical(
        fires_by_period["period"],
        categories=period_order,
        ordered=True
    )

    fires_by_period = fires_by_period.sort_values("period")

    fig_season = px.bar(
        fires_by_period,
        x="period",
        y="fire_count",
        title="Wildfire Activity by Seasonal Period",
        labels={
            "period": "Seasonal Period",
            "fire_count": "Number of Fire Detections"
        }
    )

    # ---------- FRP ----------

    frp_stats = (
        df.groupby("period")["frp"]
        .mean()
        .reset_index(name="mean_frp")
    )

    frp_stats["period"] = pd.Categorical(
        frp_stats["period"],
        categories=period_order,
        ordered=True
    )

    frp_stats = frp_stats.sort_values("period")

    fig_frp = px.bar(
        frp_stats,
        x="period",
        y="mean_frp",
        title="Average Fire Intensity by Seasonal Period",
        labels={
            "period": "Seasonal Period",
            "mean_frp": "Average Fire Radiative Power (FRP)"
        }
    )

    # ---------- KPI ----------

    total_fires = len(df_filtered)
    avg_frp = round(df_filtered["frp"].mean(),2)
    high_conf = round((df_filtered["confidence"]=="h").mean()*100,1)

    kpis = [

    html.Div([
        html.Div("Fire Detections (Filtered)", style={"fontSize":"14px"}),
        html.H3(f"{total_fires:,}", style={"margin":"5px 0"})
    ], style={
        "border":"1px solid #ddd",
        "borderRadius":"8px",
        "padding":"12px",
        "textAlign":"center",
        "flex":"1",
        "backgroundColor":"#fafafa"
    }),

    html.Div([
        html.Div("Avg FRP", style={"fontSize":"14px"}),
        html.H3(avg_frp, style={"margin":"5px 0"})
    ], style={
        "border":"1px solid #ddd",
        "borderRadius":"8px",
        "padding":"12px",
        "textAlign":"center",
        "flex":"1",
        "backgroundColor":"#fafafa"
    }),

    html.Div([
        html.Div("High Confidence", style={"fontSize":"14px"}),
        html.H3(f"{high_conf}%", style={"margin":"5px 0"})
    ], style={
        "border":"1px solid #ddd",
        "borderRadius":"8px",
        "padding":"12px",
        "textAlign":"center",
        "flex":"1",
        "backgroundColor":"#fafafa"
    })

]


    return fig_map, fig_time, fig_season, fig_frp, kpis


In [10]:
# ---------- Run Dashboard ----------

ngrok.set_auth_token("3AgCFukJVHgxVmdloDKzMOyDWDh_7xG89EE4B1W453BKwBaqV")

public_url = ngrok.connect(8050)
print("Public URL:", public_url)

app.run(port=8050)


Public URL: NgrokTunnel: "https://valerianaceous-sympathetically-apollo.ngrok-free.dev" -> "http://localhost:8050"
Dash is running on http://127.0.0.1:8050/



INFO:dash.dash:Dash is running on http://127.0.0.1:8050/



 * Serving Flask app '__main__'
 * Debug mode: off


INFO:werkzeug:WARNING: This is a development server. Do not use it in a production deployment. Use a production WSGI server instead.
 * Running on http://127.0.0.1:8050
INFO:werkzeug:Press CTRL+C to quit
INFO:werkzeug:127.0.0.1 - - [18/Mar/2026 16:55:11] "GET / HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [18/Mar/2026 16:55:11] "GET /_dash-component-suites/dash/deps/polyfill@7.v4_0_0m1773852863.12.1.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [18/Mar/2026 16:55:11] "GET /_dash-component-suites/dash/deps/react-dom@18.v4_0_0m1773852863.3.1.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [18/Mar/2026 16:55:11] "GET /_dash-component-suites/dash/deps/prop-types@15.v4_0_0m1773852863.8.1.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [18/Mar/2026 16:55:11] "GET /_dash-component-suites/dash/dash-renderer/build/dash_renderer.v4_0_0m1773852863.min.js HTTP/1.1" 200 -
INFO:werkzeug:127.0.0.1 - - [18/Mar/2026 16:55:11] "GET /_dash-component-suites/dash/dash_table/bundle.v7_0_0m17738528